# Fine-tuning a Classifier

**Fine-tuning** adapts a pre-trained model to a specific task using a small labelled dataset. Instead of training from scratch, we start from a model that already understands language, and adjust its weights on our data.

Pipeline: **load data → tokenize → load model → train with Trainer API → evaluate → save**

We use a small subset of SST-2 (Stanford Sentiment Treebank) to keep training fast.

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow accelerate evaluate scikit-learn -q

## Imports

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import load_dataset
import evaluate
import numpy as np
import torch

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

## 1. Load and Tokenize Data

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load SST-2 and take a small subset for fast demo
dataset = load_dataset("glue", "sst2")
small_train = dataset["train"].shuffle(seed=42).select(range(500))
small_val   = dataset["validation"].shuffle(seed=42).select(range(100))

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=128)

train_ds = small_train.map(tokenize, batched=True)
val_ds   = small_val.map(tokenize, batched=True)
train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Train: {len(train_ds)} examples | Val: {len(val_ds)} examples")
print(f"Example tokens: {train_ds[0]['input_ids'][:10]}...")

## 2. Load Pre-trained Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model       : {MODEL_NAME}")
print(f"Total params: {total_params:,}")
print(f"Trainable   : {trainable_params:,}")

## 3. Train with the Trainer API

The `Trainer` handles the training loop, gradient accumulation, evaluation, and checkpointing.

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
trainer.train()
print("Training complete!")

## 4. Evaluate and Test on New Sentences

In [ ]:
eval_results = trainer.evaluate()
print(f"Validation accuracy: {eval_results['eval_accuracy']:.3f}")

# Test the fine-tuned model on new sentences
from transformers import pipeline as hf_pipeline

classifier = hf_pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_sentences = [
    "This movie was absolutely wonderful!",
    "A total waste of time. I hated every minute.",
    "The acting was decent but the plot was predictable.",
]

print("\nFine-tuned model predictions:")
for sent in test_sentences:
    result = classifier(sent)[0]
    label = "POSITIVE" if result["label"] == "LABEL_1" else "NEGATIVE"
    print(f"  [{label} {result['score']:.2f}]  {sent}")

## 5. Save and Reload the Model

Save your fine-tuned model to disk and reload it for future inference — no retraining needed.

In [ ]:
save_path = "./fine_tuned_sentiment"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}/")

# Reload and verify
reloaded = hf_pipeline("text-classification", model=save_path)
print("\nReloaded model test:")
print(reloaded("The fine-tuned model works perfectly!"))